In [1]:
import os
os.chdir('/n/fs/goose/ReNO')

import torch
from IPython.display import display
from pytorch_lightning import seed_everything

import argparse
parser = argparse.ArgumentParser()

/n/fs/goose/el8403/conda-envs/reno/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/n/fs/goose/el8403/conda-envs/reno/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
import argparse
args = argparse.Namespace(
    model='sdxl-turbo',
    cache_dir='hf_cache',
    save_dir='/n/fs/goose/baseline/images',
    seed=0,
    n_inference_steps=1,
    dtype='float16',
    memsave=False,
    cpu_offloading=False,
)
print(args)

Namespace(model='sdxl-turbo', cache_dir='hf_cache', save_dir='/n/fs/goose/baseline/images', seed=0, n_inference_steps=1, dtype='float16', memsave=False, cpu_offloading=False)


In [3]:
device = torch.device('cuda')
dtype = torch.float16 if args.dtype == 'float16' else torch.float32

from models.utils import get_model
pipe = get_model(args.model, dtype, device, args.cache_dir, args.memsave, args.cpu_offloading)

shape = (1, 4, 64, 64)

WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.2.2+cu121)
    Python  3.10.14 (you have 3.10.20)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details



Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]


Loading pipeline components...:  43%|████▎     | 3/7 [00:00<00:00,  4.70it/s]


Loading pipeline components...:  57%|█████▋    | 4/7 [00:01<00:00,  3.80it/s]


Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  6.37it/s]


Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  5.61it/s]

In [4]:
# FairFace-style race categories (matches the 7-way classifier minus Middle Eastern,
# per your list) crossed with gender.
RACES = ["black", "white", "east asian", "middle eastern", "indian", "hispanic"]
GENDERS = ["man", "woman"]

PROMPT_TEMPLATE = (
    "a portrait of a {race} {gender} with a neutral expression, "
    "without any facial coverings, in a neutral white colored t-shirt against a neutral white background"
)

prompts = [
    PROMPT_TEMPLATE.format(race=race, gender=gender)
    for gender in GENDERS
    for race in RACES
]
prompts

['a portrait of a black man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a white man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a east asian man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a middle eastern man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a indian man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a hispanic man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a black woman with a neu

In [5]:
import re
import os

def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

In [6]:
import torch
from diffusers import DiffusionPipeline
from models import get_latent_shape

In [ ]:
N_SEEDS = 50

results = {}
for gender in GENDERS:
    for race in RACES:
        prompt = PROMPT_TEMPLATE.format(race=race, gender=gender)
        save_dir = os.path.join(args.save_dir, f"{slugify(race)}_{slugify(gender)}")
        os.makedirs(save_dir, exist_ok=True)

        for seed in range(N_SEEDS):
            seed_everything(seed)
            generator = torch.Generator("cuda").manual_seed(seed)
            latents = torch.randn(shape, device=device, dtype=dtype)

            with torch.no_grad():
                image = pipe.apply(
                    latents=latents,
                    prompt=prompt,
                    generator=generator,
                    num_inference_steps=args.n_inference_steps,
                )

            image_numpy = image.detach().cpu().permute(0, 2, 3, 1).float().numpy()
            image_pil = DiffusionPipeline.numpy_to_pil(image_numpy)[0]
            filename = os.path.join(save_dir, f'baseline{seed}.png')
            image_pil.save(filename)

            results[(race, gender, seed)] = filename

    print(f"done: {race} / {gender} -> {N_SEEDS} images in {save_dir}")